In [76]:
import pandas as pd
import plotly.graph_objects as go

## 0. Setup & chargement

In [77]:
df = pd.read_csv(r"..\data\processed\final.csv")
df = df.drop(columns='Unnamed: 0')
df['date'] = pd.to_datetime(df['date'])

In [78]:
df.dtypes

client_id             object
sex                   object
birth                  int64
id_prod               object
date          datetime64[ns]
session_id            object
price                float64
categ                  int64
dtype: object

In [79]:
df

,client_id,sex,birth,id_prod,date,session_id,price,categ
0,c_4410,f,1967,1_483,2021-03-13 21:35:55.949042,s_5913,15.99,1
1,c_4410,f,1967,0_1111,2021-03-22 01:27:49.480137,s_9707,19.99,0
2,c_4410,f,1967,1_385,2021-03-22 01:40:22.782925,s_9707,25.99,1
3,c_4410,f,1967,0_1455,2021-03-22 14:29:25.189266,s_9942,8.99,0
4,c_4410,f,1967,0_1420,2021-03-22 22:31:25.825764,s_10092,11.53,0
...,...,...,...,...,...,...,...,...
687529,c_84,f,1982,0_1472,2022-05-14 00:24:49.391917,s_208110,12.49,0
687530,c_84,f,1982,0_1438,2022-05-29 06:11:50.316631,s_215697,9.31,0
687531,c_84,f,1982,1_459,2022-12-17 00:16:56.629536,s_313173,15.99,1
687532,c_84,f,1982,0_1104,2022-12-17 00:24:14.357525,s_313173,13.21,0


## 1. CA dans le temps

### 1.1 CA mensuel + moyenne mobile

In [80]:
df_ca = df.set_index('date')
df_ca = df_ca.groupby(pd.Grouper(freq='ME'))['price'].sum().to_frame()
df_ca = df_ca.rename(columns={'price' : 'ca'})
df_ca['moy_mobile'] = round(df_ca['ca'].rolling(window=3).mean(), 2)
df_ca.head()

,ca,moy_mobile
date,,
2021-03-31,482440.61,NaN
2021-04-30,476109.30,NaN
2021-05-31,492943.47,483831.13
2021-06-30,484088.56,484380.44
2021-07-31,482835.40,486622.48


In [108]:
fig_ca = go.Figure()

fig_ca.add_trace(go.Scatter(
    x=df_ca.index,
    y=df_ca['ca'],
    name="CA",
    mode='lines+markers',
    marker=dict(size=5),
    line=dict(color="#58508d"),
    hovertemplate="%{y:,.0f} €"
))

fig_ca.add_trace(go.Scatter(
    x=df_ca.index,
    y=df_ca['moy_mobile'],
    name="Moyenne",
    mode='lines+markers',
    marker=dict(size=5),
    line=dict(color='#ff6361'),
    hovertemplate="%{y:,.0f} €"
))

# Calcul des bornes de l'axe Y droit pour éviter que les lignes soient écrasées
y_min = min(df_ca['ca'].min(), df_ca['moy_mobile'].min()) * 0.95
y_max = max(df_ca['ca'].max(), df_ca['moy_mobile'].max()) * 1.05

# Habillage
fig_ca.update_layout(
    yaxis=dict(title="Chiffre d'affaire", showgrid=False),
    xaxis=dict(tickformat="%b %Y",),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'),
    height=500,
    hovermode='x unified',
    hoverlabel=dict(font_size=12),
    separators=". ",
    title="Evolution du chiffre d'affaire"
)

fig_ca.show()

### 1.2 CA par catégorie

In [84]:
df_categ = df.set_index('date')
df_categ = df_categ.groupby([pd.Grouper(freq='ME'), 'categ'])['price'].sum().to_frame().reset_index()
df_categ = df_categ.rename(columns={'price' : 'ca'})
df_categ = df_categ.pivot(index='date', columns='categ', values='ca')
df_categ.head()

categ,0,1,2
date,,,
2021-03-31,193629.17,186974.17,101837.27
2021-04-30,205222.46,156138.35,114748.49
2021-05-31,196186.72,165893.40,130863.35
2021-06-30,167943.15,189162.04,126983.37
2021-07-31,144750.79,188523.27,149561.34


In [107]:
fig_categ = go.Figure()

fig_categ.add_trace(go.Bar(
    name='0',
    x=df_categ.index,
    y=df_categ[0],
    hovertemplate="%{y:,.0f} €",
    marker_color='#003f5c'
))
fig_categ.add_trace(go.Bar(
    name='1',
    x=df_categ.index,
    y=df_categ[1],
    hovertemplate="%{y:,.0f} €",
    marker_color='#bc5090'
))
fig_categ.add_trace(go.Bar(
    name='2',
    x=df_categ.index,
    y=df_categ[2],
    hovertemplate="%{y:,.0f} €",
    marker_color='#ffa600'
))


# Habillage
fig_categ.update_layout(
    yaxis=dict(title="Chiffre d'affaire", showgrid=False),
    xaxis=dict(tickformat="%b %Y",),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'),
    height=500,
    hovermode='x unified',
    hoverlabel=dict(font_size=12),
    separators=". ",
    title="Evolution du chiffre d'affaire par catégories",
    barmode='stack'
)

fig_categ.show()

### 1.3 Métriques clés / mois (clients, transactions, produits)

## 2. Zoom catalogue

### 2.1 Top / Flop produits

### 2.2 Répartition CA par catégorie

## 3. Profils clients

### 3.1 Répartition BtoB vs BtoC

### 3.2 Courbe de Lorenz